## This is only for testing purpose . 

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="mps",      # Use GPU
    torch_dtype="auto",     # Efficient memory usage
    trust_remote_code=False
)

# Let's print the model to see its layers
print(model)

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 195/195 [00:00<00:00, 737.76it/s]


Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [13]:
prompt = "The Capital of France is"

# 1. Tokenize (Text -> Numbers)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("mps")

# 2. Pass numbers through the model
model_output = model.model(input_ids)

# 3. Get the output from the Language Model Head
# This gives us scores for every possible word in the dictionary for the NEXT token
lm_head_output = model.lm_head(model_output[0])

# 4. Find the token with the highest score (argmax)
next_token_id = lm_head_output[0, -1].argmax()

# 5. Decode (Number -> Text)
next_word = tokenizer.decode(next_token_id)

print(f"Prompt: '{prompt}'")
print(f"Predicted next word: '{next_word}'")

Prompt: 'The Capital of France is'
Predicted next word: 'Paris'


In [14]:
import torch

# Get the scores for the last token position
scores = lm_head_output[0, -1]

# Get top 5 scores
top_scores, top_indices = torch.topk(scores, 5)

print("Top 5 Guesses:")
for score, idx in zip(top_scores, top_indices):
    word = tokenizer.decode(idx)
    print(f"Word: '{word}' | Score: {score.item():.2f}")

Top 5 Guesses:
Word: 'Paris' | Score: 45.50
Word: '_' | Score: 42.50
Word: 'not' | Score: 42.00
Word: 'known' | Score: 41.50
Word: ':' | Score: 41.25


In [15]:
current_input = input_ids
print(f"Starting: {prompt}", end="")

for _ in range(5):  # Let's generate 5 new words
    # Pass through model
    output = model(current_input)

    # Get next token ID
    next_token_id = output.logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)

    # Print the word
    word = tokenizer.decode(next_token_id[0])
    print(word, end="")

    # Append prediction to input for the next round
    current_input = torch.cat([current_input, next_token_id], dim=1)

Starting: The Capital of France isParis.




In [17]:
import time

long_prompt_text = "Write a very long explanation about why the sky is blue and how sunlight interacts with the atmosphere."
long_input = tokenizer(long_prompt_text, return_tensors="pt").input_ids.to("mps")

# 1. Without Cache (Slow)
start = time.time()
model.generate(long_input, max_new_tokens=20, use_cache=False)
print(f"Without Cache: {time.time() - start:.2f} seconds")

# 2. With Cache (Fast)
start = time.time()
model.generate(long_input, max_new_tokens=20, use_cache=True)
print(f"With Cache: {time.time() - start:.2f} seconds")

KeyboardInterrupt: 